# Dimensionality reduction and anomaly detection

Tables in practice have many columns, and people can only look at two or three at a time. **Dimensionality reduction** replaces many columns by a few new ones that keep as much of the relevant information as possible. This notebook covers the second block of Session 11. The first sections cover the linear standard method, **principal component analysis** (PCA), with its two key outputs, explained variance and loadings, then two non-linear methods made for pictures, **t-SNE** and **UMAP**, and what such pictures can and cannot show. The second half turns to **anomaly detection** with Isolation Forest and the local outlier factor, and ends with clusters and components used as **features** in a supervised model.

The code cells build on each other; run them in order. The first cell moves to the repository root, where the case-study paths start. They use two built-in scikit-learn datasets, 178 wines with 13 chemical measurements and 1,797 images of handwritten digits with 64 pixels each, and the Berlin Airbnb listings (prepare the data with `uv run python case-study/prepare_airbnb.py`).

```mermaid
flowchart LR
    G{"Goal?"} -->|"compress, denoise,<br/>features for a model"| PCA["PCA<br/>(linear, reversible,<br/>keeps global distances)"]
    G -->|"a 2-D picture of<br/>neighbourhoods"| NL["t-SNE or UMAP<br/>(non-linear, keeps<br/>local neighbours)"]
    PCA --> R1["Report explained variance<br/>and loadings"]
    NL --> R2["Do not read distances<br/>between clusters or cluster sizes"]
    PCA -->|"many columns:<br/>first 30-50 PCs"| NL
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Principal component analysis: explained variance and loadings

### Concept

PCA finds new axes, the **principal components** (PCs). Each PC is a weighted sum of the original columns. The first PC is the direction along which the rows vary most (largest **variance**). The second PC is the direction of largest remaining variance that is uncorrelated with (at right angles to) the first, and so on. With p columns there are p components; usually the first few carry most of the variance.

Three outputs matter:

- **Scores**: the coordinates of each row on the new axes; PC1 and PC2 scores give a 2-D map of the data.
- **Explained variance ratio**: the share of the total variance carried by each PC. A **scree plot** shows these shares; the cumulative sum tells how many PCs keep, say, 80 % of the variance.
- **Loadings**: the weights of the original columns in each PC. They say what a component *means*: a PC with large weights on alcohol, proline and colour intensity describes "strong, dark wines".

Worked example with two standardised columns that have correlation 0.8. Their covariance matrix is [[1, 0.8], [0.8, 1]]. Its **eigenvectors** (the directions that the matrix only stretches) are (1, 1)/√2 and (1, −1)/√2, with **eigenvalues** (the stretch factors, here the variances along those directions) 1 + 0.8 = 1.8 and 1 − 0.8 = 0.2. So PC1 is the average direction of the two columns and explains 1.8 / (1.8 + 0.2) = 90 % of the variance; PC2, their difference, explains 10 %. Both loadings on PC1 are 1/√2 ≈ 0.71: the two columns count equally.

![Left: scree plot for the wine data with bars falling from 36 % for PC1 to under 1 % for PC13 and a cumulative line passing 80 % at five components. Right: biplot of PC1 against PC2 with the three cultivars as separated point clouds and arrows for the six largest loadings](../figures/pca_scree_biplot.png)

The **biplot** on the right combines scores (points) and loadings (arrows). Points lie in the direction of the arrows of the columns on which they have high values. The three wine cultivars, which PCA did not see, separate clearly in the first two components.

### Why it matters

Correlated columns carry overlapping information. PCA summarises them in a few uncorrelated scores. This helps to plot high-dimensional data, to remove noise (dropping the last PCs), to speed up later methods and to deal with models that suffer from many correlated inputs. Because PCA uses variance, it must be run on **standardised** columns unless all columns share one unit; otherwise the column with the largest numbers becomes PC1 on its own.

### How it works in Python

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits, load_wine
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.preprocessing import StandardScaler

# the worked example: eigenvalues and eigenvectors of a 2 x 2 correlation matrix
eigval, eigvec = np.linalg.eigh(np.array([[1.0, 0.8], [0.8, 1.0]]))
print(eigval.round(2), np.abs(eigvec[:, -1]).round(2))   # [0.2 1.8] [0.71 0.71]

wine = load_wine(as_frame=True)                            # 178 wines, 13 measurements
Z = StandardScaler().fit_transform(wine.data)
pca = PCA().fit(Z)
print(pca.explained_variance_ratio_[:3].round(3))          # [0.362 0.192 0.111]
print(pca.explained_variance_ratio_.cumsum()[[1, 4]].round(2))   # [0.55 0.8 ]: 2 PCs 55 %, 5 PCs 80 %
print(PCA().fit(wine.data).explained_variance_ratio_[0].round(3))   # 0.998 without standardising

loadings = pd.DataFrame(pca.components_[:2].T, index=wine.data.columns, columns=["PC1", "PC2"])
print(loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(4).round(2))
#                                PC1   PC2
# flavanoids                    0.42 -0.00
# total_phenols                 0.39  0.07
# od280/od315_of_diluted_wines  0.38 -0.16
# proanthocyanins               0.31  0.04

scores = pca.transform(Z)[:, :2]                           # coordinates for a scatter plot
print(scores.shape)                                         # (178, 2)

[0.2 1.8] [0.71 0.71]
[0.362 0.192 0.111]
[0.55 0.8 ]
0.998
                               PC1   PC2
flavanoids                    0.42 -0.00
total_phenols                 0.39  0.07
od280/od315_of_diluted_wines  0.38 -0.16
proanthocyanins               0.31  0.04
(178, 2)


PC1 is a "phenol" axis (flavanoids, total phenols, proanthocyanins). Without standardising, proline (values in the hundreds and thousands) alone accounts for 99.8 % of the variance.

`PCA(n_components=0.8)` keeps as many components as needed for 80 % of the variance. The sign of a component is arbitrary: two libraries may return PC1 and −PC1.

### In practice

- **Population genetics.** Novembre et al. (2008, *Nature*): PC1 and PC2 of the genotypes of about 1,400 Europeans reproduce the map of Europe.
- **Finance.** Litterman and Scheinkman (1991, *Journal of Fixed Income*): three components, called level, slope and curvature, explain most movements of government bond yields; risk desks still report exposures on them.
- **Faces and images.** Turk and Pentland (1991) represented faces by their first principal components ("eigenfaces"); [extras notebook 10](extras/10-pca-in-depth.ipynb) (PDSH) reproduces the idea.

> **Warning.** **High explained variance is not high relevance.** PCA keeps the directions with most variance, not those most related to a target. A small component can be the one that predicts churn.

> **Caution.** **Fit PCA on training data only.** Inside a model pipeline, PCA is a preparation step like scaling: `make_pipeline(StandardScaler(), PCA(10), model)`. Fitting it on all data before a split leaks information from the test set (Session 7).

> 📖 **Read more:** [extras/09-pca-as-preprocessing.ipynb](extras/09-pca-as-preprocessing.ipynb) (INRIA MOOC): why the choice of scaling changes PCA (standard versus robust scaling, sparse and one-hot columns) and how to put PCA into a model pipeline, on the Ames housing data.
>
> 📖 **Read more:** [extras/11-pca-iris.ipynb](extras/11-pca-iris.ipynb) (scikit-learn gallery): a small first PCA: the four iris measurements projected onto three components and plotted in 3-D.

## t-SNE and UMAP for visualisation

### Concept

PCA is linear: it can only rotate and project. Data that lie on a curved surface (images of the same digit written in different ways, for example) are not separated by any flat projection. Two non-linear methods focus on **neighbourhoods** instead of variance:

- **t-SNE** (t-distributed stochastic neighbour embedding; van der Maaten and Hinton, 2008) turns distances into probabilities that two rows are neighbours, both in the original space and in a 2-D map, and moves the points in the map until the two sets of probabilities match. The **perplexity** (typically 5–50) sets roughly how many neighbours each point considers.
- **UMAP** (uniform manifold approximation and projection; McInnes, Healy and Melville, 2018) builds a graph of the **n_neighbors** nearest neighbours of every row and lays the graph out in 2-D. **min_dist** sets how tightly points may be packed. UMAP is usually faster than t-SNE, keeps somewhat more global structure and can transform new rows.

Both keep **local** structure: rows that are neighbours in the original space stay neighbours in the map. A measure of this is **trustworthiness** (between 0 and 1): how many of the 10 nearest neighbours of each point in the map are also among its true neighbours.

### Why it matters

A good 2-D map lets people see groups, outliers and mislabelled examples in data with dozens or thousands of columns (amenity indicators, TF-IDF vectors of texts in Session 13, embeddings in Session 14). But the maps distort. Distances *between* groups, the *size* of groups and the empty space in a t-SNE or UMAP map have no reliable meaning, and different settings can produce different pictures from the same data.

### How it works in Python

In [3]:
digits = load_digits()                                     # 1,797 images, 8 x 8 = 64 pixels
X = digits.data
pca_map = PCA(n_components=2).fit_transform(X)
tsne_map = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(X)   # about 10-30 s

for name, emb in {"PCA": pca_map, "t-SNE": tsne_map}.items():
    print(name, round(trustworthiness(X, emb, n_neighbors=10), 3))
# PCA 0.83
# t-SNE 0.993
print(PCA().fit(X).explained_variance_ratio_[:2].round(3))   # [0.149 0.136]: only 29 % in 2 PCs

# UMAP (package umap-learn, in the course environment)
import umap  # noqa: E402
umap_map = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(X)
print("UMAP", round(trustworthiness(X, umap_map, n_neighbors=10), 3))   # UMAP 0.988

# plot: plt.scatter(*tsne_map.T, c=digits.target, s=4, cmap="tab10")

PCA 0.83
t-SNE 0.993
[0.149 0.136]


.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP 0.988


In the t-SNE and UMAP maps the ten digits form ten separate groups, although the labels were not used; in the PCA map they overlap. [extras/13-tsne-perplexity.ipynb](extras/13-tsne-perplexity.ipynb) (scikit-learn gallery) shows how the t-SNE picture changes with the perplexity, [extras notebook 16](extras/16-umap-parameters.ipynb) (UMAP documentation) how UMAP changes with n_neighbors and min_dist.

### In practice

- **Single-cell biology.** t-SNE and UMAP maps of gene expression in thousands of single cells are the standard figure for showing cell types; Becht et al. (2019, *Nature Biotechnology*) compared UMAP with t-SNE for this use.
- **Inspecting embeddings.** The TensorFlow Embedding Projector shows PCA, t-SNE and UMAP views of word or image embeddings to explore what a model has learned.
- **Data quality.** Mislabelled training examples often appear as points of one colour inside a group of another colour in a t-SNE map of image or text data.

> **Warning.** **Do not cluster on a t-SNE map, and do not read distances between groups.** t-SNE inflates dense groups and shrinks sparse ones; distances between well-separated groups are arbitrary. Cluster on the original data (or on PCA scores) and use the map only to look.

> **Caution.** **Fix the random seed and try several settings.** t-SNE and UMAP are stochastic. A structure that appears for one perplexity or one seed only is not a finding. Wattenberg, Viégas and Johnson (2016) show striking examples.

> **Tip.** For data with hundreds of columns (text embeddings), first reduce to 30–50 principal components, then run t-SNE or UMAP on the scores. It is faster and removes noise.

> 📖 **Read more:** [extras/14-manifold-learning.ipynb](extras/14-manifold-learning.ipynb) (PDSH): non-linear embeddings before t-SNE and UMAP: multidimensional scaling, locally linear embedding and Isomap, where MDS fails, and Isomap on faces and digits.
>
> 📖 **Read more:** [extras/15-manifold-methods-compared.ipynb](extras/15-manifold-methods-compared.ipynb) (scikit-learn gallery): LLE and its variants, Isomap, MDS, spectral embedding and t-SNE side by side on the same S-shaped surface.

## What do Airbnb listings offer? PCA of amenities

### Concept

Every Airbnb listing has a list of **amenities** chosen by the host from a form: "Wifi", "Kitchen", "Hair dryer", "Wine glasses" and so on. Turned into a table, this is one 0/1 column per amenity (**indicator** or **dummy** columns): 2,653 different labels in the Berlin data, most of them rare. Such tables are wide and redundant: a listing with "Cooking basics" usually also has "Dishes and silverware" and "Dining table". PCA can show which bundles of amenities vary together and summarise a listing's equipment in a few scores.

All indicator columns share one unit (0 or 1), so PCA can be run on them without standardising; standardising would give rare amenities the same weight as common ones. Columns that almost every listing has (Wifi, 91 %) or almost none has carry little information and are dropped first.

### Why it matters

A host who wants to know "what do comparable listings offer?" or an analyst who wants to use the equipment in a price model (Sessions 6 and 10) cannot work with 2,653 columns. A few components with readable loadings are a compact description. The loadings can also reveal how the data were *collected*, which is often the more important finding.

### How it works in Python

In [4]:
import json  # noqa: E402

from sklearn.preprocessing import MultiLabelBinarizer  # noqa: E402

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
lists = listings["amenities"].apply(json.loads)                 # '["Wifi", "Kitchen", ...]' -> list
mlb = MultiLabelBinarizer()
A = pd.DataFrame(mlb.fit_transform(lists), columns=mlb.classes_, index=listings.index)
share = A.mean()
A = A.loc[:, (share >= 0.05) & (share <= 0.95)]                 # drop very rare and near-universal items
print(len(mlb.classes_), A.shape)                               # 2653 (12776, 90)

pca_a = PCA().fit(A)                                            # 0/1 columns share one unit: no scaling
print(pca_a.explained_variance_ratio_[:3].round(3),             # [0.212 0.058 0.046]
      pca_a.explained_variance_ratio_.cumsum()[[1, 9]].round(2))   # [0.27 0.48]: 2 PCs 27 %, 10 PCs 48 %
loadings = pd.DataFrame(pca_a.components_[:2].T, index=A.columns, columns=["PC1", "PC2"])
print(loadings["PC1"].nlargest(4).round(2).to_dict())
# {'Hot water kettle': 0.23, 'Dining table': 0.21, 'Cooking basics': 0.21, 'Wine glasses': 0.21}
print(loadings["PC2"].nlargest(3).round(2).to_dict(), loadings["PC2"].nsmallest(2).round(2).to_dict())
# {'Heating': 0.37, 'Stove': 0.26, 'Coffee maker': 0.23} {'Central heating': -0.25, 'Cleaning products': -0.17}

scores = pca_a.transform(A)
print(round(np.corrcoef(scores[:, 0], A.sum(axis=1))[0, 1], 2))   # 0.98: PC1 = how many amenities are listed
print(pd.crosstab(A["Heating"], A["Central heating"]))
# Central heating     0     1
# Heating
# 0                1596  2961
# 1                8219     0
medium_term = listings["minimum_nights"] >= 90
print(pd.Series(scores[:, 0]).groupby(listings["room_type"]).median().round(2).to_dict())
# {'Entire home/apt': 0.51, 'Hotel room': -0.82, 'Private room': -0.91, 'Shared room': -0.11}
print(pd.Series(scores[:, 0]).groupby(medium_term).median().round(2).to_dict())   # {False: 0.67, True: -1.59}
has_price = listings["price"].notna()
print(round(np.corrcoef(scores[has_price, 0], np.log(listings.loc[has_price, "price"]))[0, 1], 2))   # 0.34

2653 (12776, 90)
[0.212 0.058 0.046] [0.27 0.48]
{'Hot water kettle': 0.23, 'Dining table': 0.21, 'Cooking basics': 0.21, 'Wine glasses': 0.21}
{'Heating': 0.37, 'Stove': 0.26, 'Coffee maker': 0.23} {'Central heating': -0.25, 'Cleaning products': -0.17}
0.98
Central heating     0     1
Heating                    
0                1596  2961
1                8219     0
{'Entire home/apt': 0.51, 'Hotel room': -0.82, 'Private room': -0.91, 'Shared room': -0.11}
{False: 0.67, True: -1.59}
0.34


![Left: scree plot of the 90 amenity columns with a first bar of 21 % and all further bars below 6 %. Middle: the ten largest PC1 loadings, all kitchen and household items of about 0.2. Right: PC1 against PC2 scores of the listings coloured by room type; entire homes lie mostly to the right, private rooms mostly to the left, and PC2 spreads the listings vertically without separating the room types](../figures/amenities_pca.png)

Three readings. **PC1 (21 % of the variance) is the size of the amenity list**: almost all columns load positively (a handful, such as "Washer", slightly negatively), led by kitchen and household items, and the scores correlate 0.98 with the number of amenities listed. Entire homes score high, private and hotel rooms low; medium-term rentals list far fewer amenities than short stays (median −1.59 against 0.67). PC1 correlates only moderately with the log price (0.34). **PC2 is not about equipment at all.** It contrasts "Heating" with "Central heating": no listing has both labels, and listings with "Central heating" also tend to use other specific labels ("Induction stove" instead of "Stove"). They are more common among listings first reviewed since 2022. PC2 most likely separates two versions of the amenity form, a property of how the data were entered, not of the flats; the data cannot confirm this. **Two components keep only 27 % of the variance**: amenities are a list of many fairly independent items, not a few strong bundles.

### In practice

- **Survey and questionnaire analysis.** PCA (and the related factor analysis) of many yes/no or rating items is a standard way to find a few underlying dimensions, for example the OECD's PISA index of economic, social and cultural status, which for many survey cycles was the first principal component of parental education, parental occupation and home possessions.
- **Recommender systems.** Matrix factorisation of user–item tables, a close relative of PCA, was a central technique of the Netflix Prize (2006–2009).
- **Market research.** Product feature lists (cars, phones, holiday rentals) are summarised with PCA or correspondence analysis to map which products resemble each other.

> **Warning.** **A component can encode the data-collection process.** Changes of a form, a scraper or a coding scheme create correlated columns, and PCA will find them as a strong direction. Before naming a component, look at its loadings and ask whether they describe the objects or the way they were recorded.

> **Tip.** For very wide sparse tables (text, thousands of amenity labels), use `TruncatedSVD`, which works on sparse matrices without centring them; Session 13 uses it for TF-IDF vectors of texts.

*Worked example:* part B of [03-case-study-airbnb-listing-segments.ipynb](03-case-study-airbnb-listing-segments.ipynb) summarises the amenities of the Berlin listings with PCA and maps the listings with t-SNE.

## From fewer columns to unusual rows

Session 4 flagged outliers one column at a time (IQR fences, z-scores, the median absolute deviation) and in combinations of columns with the Mahalanobis distance. The next section continues with two **model-based** detectors that need no assumption of a normal distribution, **Isolation Forest** and the **local outlier factor** (LOF), and compares them with the Mahalanobis distance on the same small example. The examples are implausible Airbnb listings in Berlin (prices far from what size and location suggest, extreme minimum stays). The last section closes the session: the output of a clustering or a PCA can be fed as **features** into a supervised model, and we test whether location clusters improve a gradient-boosting price model.

```mermaid
flowchart TD
    A["Suspicious rows?"] --> B{"One column<br/>at a time?"}
    B -->|yes| U["IQR fences, robust z-score<br/>(Session 4)"]
    B -->|no| C{"Roughly elliptical<br/>(normal-like) cloud?"}
    C -->|yes| M["Robust Mahalanobis distance<br/>MinCovDet (Session 4)"]
    C -->|no| D{"Groups of different<br/>density?"}
    D -->|yes| L["Local outlier factor"]
    D -->|no| I["Isolation Forest"]
    U --> R["Rank, inspect, decide:<br/>error, rare case or new pattern"]
    M --> R
    L --> R
    I --> R
```

## Model-based anomaly detection: Isolation Forest and local outlier factor

### Concept

An **anomaly** (or outlier) is a row that differs strongly from the bulk of the data. Detectors return a **score** for every row; a threshold, or a guess of the share of anomalies (the **contamination**), turns scores into flags.

- **Isolation Forest** (Liu, Ting and Zhou, 2008) builds many random trees. Each tree splits the data on a random column at a random value, again and again, until every row stands alone. An unusual row is **isolated** after few splits, because it lies far from the others on some column. The score is the average path length to isolation over all trees: short path, high anomaly score. It needs no distances and scales to large data.
- **Local outlier factor** (Breunig et al., 2000) compares the density around a row (how close its k nearest neighbours are) with the density around those neighbours. LOF ≈ 1: as dense as its neighbourhood. LOF ≫ 1: in a much sparser region than its neighbours. Because it is *local*, it finds rows that are unusual relative to their own group, even when another group is sparse everywhere.

A small example (from Session 4). A person is 192 cm tall and weighs 55 kg. Neither value is extreme on its own (z-scores 2.2 and −1.4), but the combination is: given the height, a weight of about 88 kg would be typical.

| Method | What it looks at | Flags 192 cm / 55 kg? |
|---|---|---|
| z-score per column | each column alone | no (no \|z\| > 3) |
| robust Mahalanobis distance | distance from the centre, scaled by covariance | yes (distance 6.5 vs typical 1.2) |
| Isolation Forest, contamination 1 % | random axis-parallel splits | no (5 rows score higher) |
| LOF, 20 neighbours, contamination 1 % | density relative to neighbours | yes (LOF 4.2) |

Isolation Forest splits along one column at a time, so a point that is unusual only in the *combination* of two correlated columns is not isolated quickly. LOF and the Mahalanobis distance see the combination.

### Why it matters

Data errors, fraud, machine faults and genuinely new patterns all appear as anomalies. Univariate rules miss unusual combinations, and the Mahalanobis distance assumes one elliptical cloud. Model-based detectors cover the other cases. No detector is best everywhere; [extras/17-anomaly-detection-compared.ipynb](extras/17-anomaly-detection-compared.ipynb) (scikit-learn gallery) shows four detectors on five toy datasets, and [extras notebook 20](extras/20-outlier-detection-benchmark.ipynb) (scikit-learn gallery) compares Isolation Forest and LOF on real benchmark data.

### How it works in Python

In [5]:
import numpy as np
import pandas as pd
from scipy.stats import chi2
from sklearn.covariance import MinCovDet
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
height = rng.normal(172, 9, 500)                                   # cm
weight = 0.9 * height - 85 + rng.normal(0, 6, 500)                 # kg, correlated with height
X = np.vstack([np.column_stack([height, weight]), [192, 55]])      # row 500: tall and light

print(((X[500] - X.mean(axis=0)) / X.std(axis=0)).round(1))       # [ 2.2 -1.4]  no |z| > 3
d2 = MinCovDet(random_state=0).fit(X).mahalanobis(X)               # squared robust distances
print(np.sqrt(d2[500]).round(1), (d2 > chi2.ppf(0.999, df=2)).sum())   # 6.5 3

iso = IsolationForest(contamination=0.01, random_state=0).fit(X)
iso_score = -iso.score_samples(X)                                  # higher = more anomalous
print(iso.predict(X)[500], (iso_score > iso_score[500]).sum())    # 1 5   (1 = inlier, -1 = outlier)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
flags = lof.fit_predict(X)
print(flags[500], round(-lof.negative_outlier_factor_[500], 2))   # -1 4.22

[ 2.2 -1.4]


6.5 3


1 5
-1 4.22


Now on real data: the Berlin Airbnb listings. A platform, a city office or a data analyst wants to find **implausible listings**: prices far from what the size and location suggest, extreme minimum stays, typing errors. We use the 6,924 listings with a price and a minimum stay below 90 nights; the medium-term rentals found in the lecture of block 1 have a different kind of price and are left out.

In [6]:
from sklearn.ensemble import HistGradientBoostingRegressor  # noqa: E402
from sklearn.model_selection import KFold, cross_val_predict  # noqa: E402

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 12)
listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
lat0, lon0 = 52.5219, 13.4132                                   # Alexanderplatz
listings["north_km"] = (listings["latitude"] - lat0) * 111.2
listings["east_km"] = (listings["longitude"] - lon0) * 111.2 * np.cos(np.radians(lat0))
listings["km_centre"] = np.hypot(listings["north_km"], listings["east_km"])
short = listings[listings["price"].notna() & (listings["minimum_nights"] < 90)].reset_index(drop=True)
short["log_price"] = np.log(short["price"])
short["log_min_nights"] = np.log(short["minimum_nights"])
print(len(short))                                               # 6924

cols = ["log_price", "accommodates", "log_min_nights", "km_centre"]
Z = StandardScaler().fit_transform(short[cols])
short["iso"] = -IsolationForest(random_state=0).fit(Z).score_samples(Z)
short["lof"] = -LocalOutlierFactor(n_neighbors=20).fit(Z).negative_outlier_factor_
show = ["property_type", "accommodates", "price", "minimum_nights", "km_centre"]
print(short.nlargest(3, "iso")[show + ["iso"]].round(2))
#      property_type  accommodates   price  minimum_nights  km_centre   iso
# 4689     Houseboat            16  4458.0             1.0      22.45  0.75   <- large, expensive, far out
# 5497   Entire home            16  2296.0             1.0      21.07  0.74
# 5826     Houseboat            12  2270.4             1.0      22.43  0.72
print(short.nlargest(3, "lof")[show + ["lof"]].round(2))
#              property_type  accommodates    price  minimum_nights  km_centre   lof
# 2405    Entire rental unit             3    72.28            60.0       6.86  2.89   <- 60-night minimum
# 5410  Shared room in hotel            16    46.00             1.0       3.49  2.61   <- 16 guests for 46 euros
# 5497           Entire home            16  2296.00             1.0      21.07  2.52

6924


     property_type  accommodates   price  minimum_nights  km_centre   iso
4689     Houseboat            16  4458.0             1.0      22.45  0.75
5497   Entire home            16  2296.0             1.0      21.07  0.74
5826     Houseboat            12  2270.4             1.0      22.43  0.72
             property_type  accommodates    price  minimum_nights  km_centre   lof
2405    Entire rental unit             3    72.28            60.0       6.86  2.89
5410  Shared room in hotel            16    46.00             1.0       3.49  2.61
5497           Entire home            16  2296.00             1.0      21.07  2.52


Isolation Forest ranks the listings at the edge of the data first: houseboats and large houses for 12–16 guests at more than €2,000 a night, 21–22 km from the centre. They are rare, but not necessarily wrong. LOF finds listings that do not fit their neighbours: a 60-night minimum among short stays, a shared room in a hotel for 16 guests at €46 (a dormitory bed, probably, with the room size as "guests"). A sharper check states directly what "implausible" means: **the price compared with the price expected for the size, room type and location**. The expected price comes from a regression model (Sessions 6 and 10); each listing's prediction is made by a model that did not see that listing (`cross_val_predict`), so a listing cannot explain away its own price.

In [7]:
X = pd.get_dummies(short[["accommodates", "bedrooms", "east_km", "north_km", "room_type"]], dtype=float)
expected = cross_val_predict(HistGradientBoostingRegressor(random_state=0), X, short["log_price"],
                             cv=KFold(5, shuffle=True, random_state=0))
short["ratio"] = np.exp(short["log_price"] - expected)          # actual price / expected price
print(short["ratio"].quantile([0.01, 0.5, 0.99]).round(2).to_list())   # [0.37, 1.0, 2.86]
print(((short["ratio"] > 3) | (short["ratio"] < 1 / 3)).sum())         # 110 listings off by a factor of 3
print(short.nlargest(3, "ratio")[show + ["number_of_reviews", "ratio"]].round(2))
#      property_type  accommodates    price  minimum_nights  km_centre  number_of_reviews  ratio
# 6858   Entire home             5   7999.2             1.0      15.60                  0  40.79
# 4024   Entire loft             7  10025.0             2.0      14.50                 68  34.32
# 5406   Entire home             2    950.0             1.0       9.64                 11   9.66
print(short.nsmallest(3, "ratio")[show + ["number_of_reviews", "ratio"]].round(2))
#                     property_type  accommodates  price  minimum_nights  km_centre  number_of_reviews  ratio
# 95             Entire rental unit             2  14.68             1.0       2.94                 51   0.09
# 70    Private room in rental unit             2   9.03            25.0       3.59                310   0.09
# 1858           Entire rental unit             2  18.85             2.0       4.38                 24   0.10
print((listings["minimum_nights"] > 365).sum(), listings["minimum_nights"].max())   # 10 1125.0

top_iso = set(short.nlargest(100, "iso").index)
top_ratio = set(short.assign(dev=np.abs(np.log(short["ratio"]))).nlargest(100, "dev").index)
print(len(top_iso & top_ratio))                                 # 27: the two lists overlap little

[0.37, 1.0, 2.86]
110
     property_type  accommodates    price  minimum_nights  km_centre  number_of_reviews  ratio
6858   Entire home             5   7999.2             1.0      15.60                  0  40.79
4024   Entire loft             7  10025.0             2.0      14.50                 68  34.32
5406   Entire home             2    950.0             1.0       9.64                 11   9.66
                    property_type  accommodates  price  minimum_nights  km_centre  number_of_reviews  ratio
95             Entire rental unit             2  14.68             1.0       2.94                 51   0.09
70    Private room in rental unit             2   9.03            25.0       3.59                310   0.09
1858           Entire rental unit             2  18.85             2.0       4.38                 24   0.10
10 1125.0
27


Half of the listings cost between 0.81 and 1.25 times their expected price (the median ratio is 1.0); 110 listings are off by a factor of three or more. At the top, a flat for five at €7,999 a night (40 times the expected price) and the €10,025 loft for seven: placeholder prices that block the calendar or typing errors, not market prices. At the bottom, entire flats near the centre for €15–19 a night, a tenth of the expected price: probably monthly rates entered as nightly prices, or discounts that the scraper read as the price. Ten listings in the whole table require a minimum stay of more than a year, up to 1,125 nights; for a short-term rental platform that is a placeholder, not a rule. Only 27 of the top 100 Isolation Forest listings are among the 100 largest price ratios: a generic detector looks for *rare* listings, the domain check for *inconsistent* ones. The domain check is easier to explain and to act on ("price 40 times the expected price; please check"), and it says what to do next: a person checks the listing page, and the price model of Session 10 excludes or caps such rows.

### In practice

- **Fraud detection.** Card issuers and payment providers screen transactions with anomaly scores alongside supervised fraud models, because new fraud patterns have no labels yet. The credit-card fraud data of the Université Libre de Bruxelles (Dal Pozzolo et al., 2015), a common public benchmark, has 0.17 % fraudulent transactions.
- **Industrial monitoring.** Condition monitoring of machines (bearings, turbines) scores multivariate sensor readings and raises alarms for review, because faults are rare and varied.
- **IT operations.** Monitoring platforms flag unusual combinations of latency, error rate and traffic in server metrics; an engineer then decides whether there is an incident.

> **Warning.** **The contamination parameter is a guess, not a result.** `contamination=0.01` simply flags the top 1 % of scores. Prefer to look at the ranked scores and choose a threshold with the people who will review the flags (how many can they check per week?).

> **Caution.** **Anomalous is not the same as wrong.** The top Isolation Forest listings above are houseboats and large houses that may well be genuine. Deleting flagged rows without inspection removes exactly the cases that may matter most (Session 4).

> **Tip.** Without labels, evaluate a detector by inspecting a sample of the top-ranked rows. With a few known anomalies, use precision among the top k or the ROC AUC, as in [extras notebook 20](extras/20-outlier-detection-benchmark.ipynb).

## Clusters and components as features

### Concept

Unsupervised outputs can be inputs to a supervised model:

- **Cluster membership** as a categorical feature (one-hot encoded), or the **distances to each centroid** (`KMeans.transform`), which are numeric and smoother.
- **Principal component scores** instead of, or in addition to, many correlated columns.
- **Anomaly scores** as a feature ("how unusual is this transaction?").
- **Aggregates at another level**, such as the cluster of a whole group of rows (all listings of a host, all products of a seller) attached to each row.

Like scaling, these are **preparation steps** that are fitted. They belong inside the pipeline and are fitted on the training folds only. Clusters computed on all data, including the test rows, leak information.

```mermaid
flowchart LR
    T["Training rows"] --> S["Scaler.fit"]
    S --> K["KMeans.fit<br/>(or PCA.fit)"]
    K --> F["Original features +<br/>centroid distances"]
    F --> M["Model.fit"]
    V["Validation rows"] -.->|"transform only"| S
    S -.-> K
    K -.-> F
    F -.->|"predict"| M
```

### Why it matters

A cluster or a component can summarise many columns in a way a model would otherwise have to learn from scratch, which helps simple models and small data. With flexible models (gradient boosting) and enough data, the gain is often small, because the model can find the same structure itself. The only way to know is to compare with and without, using cross-validation or a time split.

### How it works in Python

In [8]:
from sklearn.cluster import KMeans  # noqa: E402
from sklearn.compose import make_column_transformer  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score  # noqa: E402
from sklearn.pipeline import FunctionTransformer, make_pipeline, make_union  # noqa: E402
from sklearn.preprocessing import OrdinalEncoder  # noqa: E402

url = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
telco = pd.read_csv(url)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")
telco = telco.dropna(subset=["TotalCharges"])
tcols = ["tenure", "MonthlyCharges", "TotalCharges"]
y = (telco["Churn"] == "Yes").astype(int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
models = {
    "3 columns": make_pipeline(StandardScaler(), LogisticRegression()),
    "3 columns + 8 centroid distances": make_pipeline(
        StandardScaler(),
        make_union(FunctionTransformer(), KMeans(n_clusters=8, n_init=10, random_state=0)),
        LogisticRegression(max_iter=1000)),
    "2 principal components": make_pipeline(StandardScaler(), PCA(n_components=2), LogisticRegression()),
}
for name, model in models.items():
    auc = cross_val_score(model, telco[tcols], y, cv=cv, scoring="roc_auc").mean()
    print(f"{name:34s} ROC AUC {auc:.3f}")
# 3 columns                          ROC AUC 0.809
# 3 columns + 8 centroid distances   ROC AUC 0.814
# 2 principal components             ROC AUC 0.807

3 columns                          ROC AUC 0.809


3 columns + 8 centroid distances   ROC AUC 0.814
2 principal components             ROC AUC 0.807


`make_union(FunctionTransformer(), KMeans(...))` keeps the original columns and appends the eight distances; because it sits inside the pipeline, k-means is refitted in every fold. The centroid distances let the linear model bend its boundary and gain 0.005 ROC AUC, a small effect of the size of the fold-to-fold noise. Two components lose almost nothing compared with three columns.

Now a case where a cluster feature has a plausible job: **location in a price model**. A gradient-boosting model for the log price of the short-stay listings (as in Session 10) knows the district, but Berlin's twelve districts are large and mixed. k-means on the coordinates gives 20 **location clusters**, compact areas of similar size; their centroid distances describe where a listing is more finely than the district. The baseline is the district alone, the competitor the raw coordinates, which a tree can split on directly. Hosts with many similar listings would make random folds optimistic (Session 7), so the folds are grouped by host.

In [9]:
def price_model(location=None):
    parts = [(OrdinalEncoder(), ["room_type", "district"]),
             ("passthrough", ["accommodates", "bedrooms", "minimum_nights"])]
    if location == "clusters":                                   # 20 centroid distances, refitted per fold
        parts.append((KMeans(n_clusters=20, n_init=10, random_state=0), ["east_km", "north_km"]))
    if location == "coordinates":
        parts.append(("passthrough", ["east_km", "north_km"]))
    return make_pipeline(make_column_transformer(*parts),
                         HistGradientBoostingRegressor(categorical_features=[0, 1], random_state=0))


for name, location in [("district only", None), ("+ 20 location-cluster distances", "clusters"),
                       ("+ raw coordinates", "coordinates")]:
    mae_log = -cross_val_score(price_model(location), short, short["log_price"], groups=short["host_id"],
                               cv=GroupKFold(n_splits=5), scoring="neg_mean_absolute_error")
    print(f"{name:32s} MAE (log) {mae_log.mean():.3f} +/- {mae_log.std():.3f}")
# district only                    MAE (log) 0.299 +/- 0.008
# + 20 location-cluster distances  MAE (log) 0.291 +/- 0.009
# + raw coordinates                MAE (log) 0.289 +/- 0.010

district only                    MAE (log) 0.299 +/- 0.008


+ 20 location-cluster distances  MAE (log) 0.291 +/- 0.009


+ raw coordinates                MAE (log) 0.289 +/- 0.010


An MAE of 0.30 on the log scale means that a typical prediction is off by a factor of about e^0.30 ≈ 1.35. The location clusters reduce the error a little (0.299 to 0.291, about the size of the fold-to-fold spread), but the raw coordinates do as well or slightly better (0.289): a tree can find the expensive areas itself when it gets the coordinates. Cluster features earn their place where the model cannot use the raw information (a linear model cannot use latitude and longitude well), where a summary must be explained to people ("listings in location cluster 7"), or where the same areas are reused in many analyses.

### In practice

- **Feature learning with k-means.** Coates and Ng (2012) showed that centroid distances of k-means fitted on image patches give features that are competitive with much more complex methods for image recognition.
- **Principal component regression** replaces many correlated predictors by their first components (ISLP, Chapter 6.3); it is used in chemometrics, where spectra have hundreds of correlated wavelengths.
- **Geodemographic codes as covariates.** Area classifications such as the ONS Output Area Classification are used as area-level variables in health and social research.

> **Warning.** **Fit the clustering inside the cross-validation.** Clusters fitted on all rows, then used as a feature in cross-validation, carry information from the validation folds. Use a pipeline, as in both examples above, or build the clusters from an earlier period only.

> **Caution.** **A small gain in one split is not evidence.** Report the spread across folds or a confidence interval (Session 7) before claiming that clusters improve a model. And compare with the obvious alternative: here, raw coordinates.

*Worked example:* part C of [03-case-study-airbnb-listing-segments.ipynb](03-case-study-airbnb-listing-segments.ipynb) ranks the Berlin listings with Isolation Forest, LOF and the price ratio, inspects the top of each list, and compares the price model of Session 10 with and without location clusters under grouped cross-validation.

## Check your understanding

1. Two standardised columns have correlation 0. What are the eigenvalues of their correlation matrix, and how much variance does PC1 explain?
2. In the wine data PC1 explains 36 % after standardising and 99.8 % without. Which result is misleading, and why?
3. What does a loading of 0.42 for flavanoids on PC1 tell you? What does it not tell you?
4. In a t-SNE map, group A is twice as large as group B and far away from it. Which of these two observations can you report?
5. When would you prefer PCA scores over a UMAP map as input to a later model?
6. The second amenity component contrasts "Heating" with "Central heating". Why should you not call it a "heating quality" component? How could you check the explanation offered above?
7. Why does Isolation Forest miss the 192 cm / 55 kg person while LOF finds it?
8. A colleague sets `contamination=0.05` and reports "5 % of the Berlin listings are fake". What is wrong with this statement?
9. The top Isolation Forest listings are houseboats and large houses far from the centre; the top price-ratio listings cost 30–40 times their expected price. Which list would you send to a city office, and why?
10. Why must the expected price be predicted by a model that did not see the listing (`cross_val_predict`) rather than by a model fitted on all listings?
11. Location clusters lower the MAE from 0.299 to 0.291, raw coordinates to 0.289. What would you recommend for the Session 10 price model, and when would the clusters still be useful?
12. A listing has a price ratio of 0.1: it costs a tenth of what its size and location suggest. Name three possible explanations and how you would tell them apart.

## Further reading

- James, G., Witten, D., Hastie, T., Tibshirani, R. and Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*, Chapter 12.2 "Principal Components Analysis". Springer. https://www.statlearning.com/
- Wattenberg, M., Viégas, F. and Johnson, I. (2016). How to use t-SNE effectively. *Distill*. https://doi.org/10.23915/distill.00002
- Coenen, A. and Pearce, A. (2019). *Understanding UMAP*. Google PAIR. https://pair-code.github.io/understanding-umap/
- McInnes, L., Healy, J. and Melville, J. (2018). UMAP: uniform manifold approximation and projection for dimension reduction. arXiv:1802.03426. https://arxiv.org/abs/1802.03426
- scikit-learn developers (2026). *User Guide: 2.7 Novelty and Outlier Detection*. https://scikit-learn.org/stable/modules/outlier_detection.html
- Liu, F. T., Ting, K. M. and Zhou, Z.-H. (2008). Isolation forest. *Proceedings of the 8th IEEE International Conference on Data Mining*, 413–422. https://doi.org/10.1109/ICDM.2008.17
- Breunig, M. M., Kriegel, H.-P., Ng, R. T. and Sander, J. (2000). LOF: identifying density-based local outliers. *Proceedings of ACM SIGMOD 2000*, 93–104. https://doi.org/10.1145/342009.335388
- Hyndman, R. J. (2026). *That's Weird: Anomaly Detection Using R*. OTexts. https://otexts.com/weird/

> 📖 **Read more:** [extras/21-islp-unsupervised-lab.ipynb](extras/21-islp-unsupervised-lab.ipynb) (ISLP labs): the Chapter 12 lab in Python, starting with PCA of the USArrests data (scores, loadings, scree plot) and matrix completion. Needs the package `ISLP`.